# H SC 790 · Lab 10 · Train Here, Validate There, Audit Everyone

**Week 11.** Due Thursday, April 15, 11:59 PM · 30 points.

**What earns full credit**
- Runs end to end from a fresh runtime
- Calibration shown before and after recalibration
- Subgroup audit with intervals
- Label comparison and AI-use note

**Rules.** Everything here is synthetic. Never load protected health information or confidential employer data into Colab or GitHub. The lab is finished when it runs top to bottom from a fresh runtime (Runtime → Restart session and run all).

The data dictionary for this lab is the README in the lab's folder of the course repository.

Cells marked `# YOUR CODE HERE` and *Your answer here.* are yours to complete. Add cells freely.

In [ ]:
# Setup: run this cell first, every time.
import os, json, urllib.request
import numpy as np
import pandas as pd

# The course data, read straight from the course repository. Never from your own machine.
COURSE_URL = os.environ.get("HSC790_DATA", "https://raw.githubusercontent.com/lmlevine/hsc790-spr27/main/data/")

def read_text(rel):
    """Text of one file in the course data folder, for example read_text('SNAPSHOT.txt')."""
    return urllib.request.urlopen(COURSE_URL + rel).read().decode()

print("Data snapshot:", read_text("SNAPSHOT.txt").strip())

Outreach staff can reach a limited number of adults with type 2 diabetes each year and want those most likely to have an avoidable emergency visit in the next 12 months. Build a risk model at Site A (PCHS), test it unchanged at Site B (Lucero), recalibrate it, audit it by subgroup, and then see what changes when the label changes.

Read the "When recorded" column of the data dictionary (the README in `data/lab10/`) before you choose predictors.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

A = pd.read_csv(COURSE_URL + "lab10/site_a.csv", parse_dates=["prediction_date"])
B = pd.read_csv(COURSE_URL + "lab10/site_b.csv", parse_dates=["prediction_date"])
rng = np.random.default_rng(790)
print("Site A", A.shape, "outcome rate", A.avoidable_ed_12m.mean().round(3), "| Site B", B.shape, "outcome rate", B.avoidable_ed_12m.mean().round(3))
A.head(3)

## 1. Fix the label and the prediction moment

Write both down. Then, for each candidate predictor, say whether it is known at the prediction moment, and exclude what is not.

*Your answer here.*

In [ ]:
def features(df, extra=()):
    """The model matrix. Decide what goes in, and how missing values are handled."""
    # YOUR CODE HERE
    pass

features(A).describe().T[["mean", "min", "max"]].round(2)

## 2. Train at Site A

Logistic regression with a patient-level, time-ordered split: no member on both sides, training rows earlier than test rows. Report internal AUC and a calibration plot. Then, once, add the predictor you excluded and see what it does to the AUC.

In [ ]:
# YOUR CODE HERE

## 3. Validate at Site B

Apply the model unchanged. Report AUC, calibration intercept and slope, and the plot. What changed? Compare the predictors' distributions at the two sites before you explain it.

In [ ]:
B["y"] = B.avoidable_ed_12m
B["group"] = B.region
# --- score B unchanged (slide) ---
B['p'] = m.predict_proba(features(B))[:, 1]
print('AUC at B:', roc_auc_score(B.y, B.p))

# --- calibration intercept and slope at B (slide) ---
lp = np.log(B.p / (1 - B.p)).to_frame('lp')
cal = LogisticRegression(C=1e6).fit(lp, B.y)
print('slope', cal.coef_[0, 0], 'int', cal.intercept_[0])

# YOUR CODE HERE

## 4. Recalibrate

Update the intercept, then the intercept and slope, using part of Site B. Re-measure on the rest.

In [ ]:
# YOUR CODE HERE

## 5. Audit by subgroup

At your operating threshold: outcome rate, AUC, sensitivity, predictive value, and share flagged, by group, with intervals. The slide's summary is below with `group` set to region; add intervals, and repeat for another grouping.

In [ ]:
t = B.p.quantile(0.8)          # an operating threshold: outreach capacity for the top 20%
# --- subgroup audit at threshold t (slide) ---
B['flag'] = B.p >= t
audit = B.groupby('group').apply(lambda g: pd.Series({
    'n': len(g), 'rate': g.y.mean(),
    'auc': roc_auc_score(g.y, g.p),
    'sens': g.flag[g.y == 1].mean(),
    'ppv': g.y[g.flag].mean(),
    'flagged': g.flag.mean()}), include_groups=False)
audit.round(3)

In [ ]:
# YOUR CODE HERE

## 6. Swap the label

Retrain to predict cost instead (`cost_12m`). Compare who is selected for outreach under each label: the top 10% by each model. `active_conditions_12m` measures health need in the outcome year; use it to compare how sick the selected people are.

In [ ]:
# YOUR CODE HERE

## Interpretation

Cite your own output in each answer.

**1. What happened at Site B?** Discrimination and calibration, before and after recalibration.

*Your answer here.*

**2. For whom is the model worst?** Name the subgroup, the measure, and the size of the gap with its interval.

*Your answer here.*

**3. What did the label change?** Compare who is selected under cost and under avoidable visits.

*Your answer here.*

**4. Would you deploy it?** State the conditions, the owner, and what would make you switch it off.

*Your answer here.*

## AI-use note (required)

Say which AI tools you used, for what, and one thing you checked or corrected in their output. "None" is an acceptable answer.

*Your note here.*

## Submit

1. Runtime → Restart session and run all. Fix anything that fails.
2. File → Save a copy in GitHub, into `labs/lab10/` of your course repository.